# Instalaciones y dependencias

In [8]:
!pip install pandas langchain langchain-experimental tabulate langchain-ollama

# *Importamos las librerias*

In [9]:
import pandas as pd
import langchain

print("Pandas version:", pd.__version__)
print("LangChain version:", langchain.__version__)
print(" Todo instalado correctamente")

Pandas version: 2.2.2
LangChain version: 1.2.15
 Todo instalado correctamente


# ***Configuramos el API Key***

In [10]:
from google.colab import userdata
import os

os.environ["MISTRAL_API_KEY"] = userdata.get("MISTRAL_API_KEY")
print("API Key cargada correctamente")

# ttlUH11iq9d1roD58pA3gOlq4DC0zPet

API Key cargada correctamente


# Instalar Mistral


In [11]:
!pip install langchain-mistralai

In [27]:
from langchain_mistralai import ChatMistralAI

llm = ChatMistralAI(
    model="mistral-small-latest",
    temperature=0
)

respuesta = llm.invoke("Respondé solo esto: ¿estás funcionando?")
print(respuesta.content)

Sí, estoy funcionando correctamente. ¿En qué puedo ayudarte?


# **Cargar el dataset y convertirlo en documentos**

In [16]:
# Instalamos librerías necesarias para el agente RAG:
!pip install langchain-mistralai langchain-community faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 23.4 MB/s eta 0:00:00


In [17]:
# Instalamos la librería principal LangChain, que provee las herramientas base
# para construir agentes RAG, manejar prompts, cadenas, retrievers y conectores con LLMs.
!pip install langchain

In [18]:
import pandas as pd
from langchain_core.documents import Document


# --- Cargamos el archivo ---
df = pd.read_csv("data_sample.csv")
print(f"Dataset cargado: {df.shape[0]} filas x {df.shape[1]} columnas")
print(df.head(3))

# --- Convertir cada fila en un Document ---
# Cada Document es como una "página" que el sistema puede buscar
def df_a_documentos(df):
    docs = []
    for idx, fila in df.iterrows():
        # Convierte la fila en texto: "columna: valor"
        texto = "\n".join(f"{col}: {val}" for col, val in fila.items() if pd.notna(val))
        docs.append(Document(page_content=texto, metadata={"fila": idx}))
    return docs

documentos = df_a_documentos(df)
print(f"Documentos creados: {len(documentos)}")
print("\nEjemplo del primero:")
print(documentos[0].page_content)

Dataset cargado: 2823 filas x 25 columnas
   ORDERNUMBER  QUANTITYORDERED  PRICEEACH  ORDERLINENUMBER    SALES  \
0        10107               30      95.70                2  2871.00   
1        10121               34      81.35                5  2765.90   
2        10134               41      94.74                2  3884.34   

        ORDERDATE   STATUS  QTR_ID  MONTH_ID  YEAR_ID  ...  \
0  2/24/2003 0:00  Shipped       1         2     2003  ...   
1   5/7/2003 0:00  Shipped       2         5     2003  ...   
2   7/1/2003 0:00  Shipped       3         7     2003  ...   

                    ADDRESSLINE1  ADDRESSLINE2   CITY STATE POSTALCODE  \
0        897 Long Airport Avenue           NaN    NYC    NY      10022   
1             59 rue de l'Abbaye           NaN  Reims   NaN      51100   
2  27 rue du Colonel Pierre Avia           NaN  Paris   NaN      75508   

  COUNTRY TERRITORY CONTACTLASTNAME CONTACTFIRSTNAME DEALSIZE  
0     USA       NaN              Yu             Kwai    Sma

In [19]:
# Instalamos todas las librerías necesarias para construir el agente RAG:
# - langchain: núcleo principal para manejar prompts, cadenas y retrievers.
# - langchain-community: integraciones adicionales (ej. FAISS).
# - langchain-mistralai: conexión con los modelos de Mistral AI (embeddings y chat).
# - faiss-cpu: motor de búsqueda vectorial rápido para indexar y recuperar documentos.
!pip install langchain langchain-community langchain-mistralai faiss-cpu

#Crear el Vector Store (la "memoria buscable")

In [20]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_mistralai import MistralAIEmbeddings
from langchain_community.vectorstores import FAISS


# --- Paso 2a: Dividir documentos en chunks ---
# Necesario porque algunos documentos pueden ser muy largos
# Con 25 columnas por fila, cada doc tiene ~500 caracteres → chunk_size=600 es suficiente
splitter = RecursiveCharacterTextSplitter(
    chunk_size=600,
    chunk_overlap=50   # overlap evita que se pierda contexto entre chunks
)
chunks = splitter.split_documents(documentos)
print(f"Chunks generados: {len(chunks)}")

# --- Paso 2b: Generar embeddings ---
# Un "embedding" convierte texto en números que representan su significado
# Así FAISS puede comparar preguntas con filas del dataset
embeddings = MistralAIEmbeddings(model="mistral-embed")

# --- Paso 2c: Crear el índice FAISS ---
# Esto puede tardar 1-2 minutos con 2823 filas (hace llamadas a la API)
print("Creando vector store... (puede tardar un poco)")
vectorstore = FAISS.from_documents(chunks, embeddings)

# --- Paso 2d: Crear el retriever ---
# k=6 significa: recuperar las 6 filas más similares a la pregunta
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 6}
)
print("✅ Vector store listo")

# --- Verificación: probá buscar algo manualmente ---
resultados = retriever.invoke("ventas de motocicletas en USA")
print(f"\nResultados de prueba ({len(resultados)} encontrados):")
for r in resultados[:2]:
    print("---")
    print(r.page_content[:200])

Chunks generados: 2823


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer.json: 0.00B [00:00, ?B/s]

Creando vector store... (puede tardar un poco)
✅ Vector store listo

Resultados de prueba (6 encontrados):
---
ORDERNUMBER: 10211
QUANTITYORDERED: 46
PRICEEACH: 54.09
ORDERLINENUMBER: 8
SALES: 2488.14
ORDERDATE: 1/15/2004 0:00
STATUS: Shipped
QTR_ID: 1
MONTH_ID: 1
YEAR_ID: 2004
PRODUCTLINE: Motorcycles
MSRP: 6
---
ORDERNUMBER: 10402
QUANTITYORDERED: 59
PRICEEACH: 87.6
ORDERLINENUMBER: 3
SALES: 5168.4
ORDERDATE: 4/7/2005 0:00
STATUS: Shipped
QTR_ID: 2
MONTH_ID: 4
YEAR_ID: 2005
PRODUCTLINE: Motorcycles
MSRP: 76
P


# Armar la cadena RAG y hacer preguntas

In [21]:
!pip install langchain langchain-core

In [22]:
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# CELDA 4 — Cadena RAG
PROMPT_TEMPLATE = """
Sos un analista de ventas experto. Respondé usando ÚNICAMENTE los datos del contexto.
Si no está en el contexto, decí "No tengo suficientes datos para responder eso."

Contexto:
{context}

Pregunta: {question}

Respuesta:"""

prompt = PromptTemplate(input_variables=["context", "question"], template=PROMPT_TEMPLATE)

def formatear_docs(docs):
    return "\n\n---\n".join(doc.page_content for doc in docs)

cadena_rag = (
    {"context": retriever | formatear_docs, "question": RunnablePassthrough()}
    | prompt
    | llm   # recordá definir tu modelo de lenguaje, ej: ChatMistral
    | StrOutputParser()
)

def consultar(pregunta):
    print(f" {pregunta}")
    print(f"\n {cadena_rag.invoke(pregunta)}")
    print("─" * 50)

print(" Agente RAG listo")


 Agente RAG listo


In [23]:
consultar("¿Cuál es el ticket promedio de ventas por país?")
consultar("¿Qué país tiene la mayor cantidad de pedidos en estado Shipped?")
consultar("¿Cuál es la línea de productos con más ventas en cada país?")
consultar("¿Cuál es el precio unitario promedio por línea de producto?")
consultar("¿Cómo evolucionan las ventas mes a mes?")
consultar("¿Qué cliente realizó el pedido con el monto total más alto?")
consultar("¿Cuál es el país con el ticket promedio más alto?")
consultar("¿Cuántos pedidos hay en estado Processing vs Cancelled?")
consultar("¿Cuál es la temporada con más ventas?")


 ¿Cuál es el ticket promedio de ventas por país?

 El único país presente en los datos es **España**, con las siguientes ventas totales:

- **10197**: 1209.59
- **10350**: 6490.88
- **10424**: 7969.36
- **10104**: 3227.63
- **10153**: 1779.71
- **10386**: 3006.12

**Total de ventas en España**: 23,683.29
**Número de órdenes**: 6

**Ticket promedio en España**: **3,947.22** (23,683.29 / 6).
──────────────────────────────────────────────────
 ¿Qué país tiene la mayor cantidad de pedidos en estado Shipped?

 En el contexto proporcionado, **España** es el único país con pedidos en estado *"Shipped"* (todos los registros corresponden a España).

No hay datos de otros países para comparar.
──────────────────────────────────────────────────
 ¿Cuál es la línea de productos con más ventas en cada país?

 No tengo suficientes datos para responder eso. El contexto solo incluye información de ventas para **España**.
──────────────────────────────────────────────────
 ¿Cuál es el precio unitario pr